# Comparaison des schémas mensuels (janvier à mars 2024)

## Objectif

Vérifier que les fichiers de janvier, février et mars 2024 ont le même schéma avant de les charger dans la même table Bronze.

## Pourquoi cette vérification

La couche Bronze utilise **une seule table** pour tous les mois (décision de conception). Cela suppose que les fichiers ont les mêmes colonnes et les mêmes types. Si la source ajoute une colonne ou change un type, un chargement par nom de colonne peut échouer ou, plus grave, convertir mal les valeurs. Le chargement de janvier l'a déjà montré : les horodatages avaient été mal lus par défaut.

## Méthode

On lit uniquement les **métadonnées** de chaque fichier Parquet (schéma et nombre de lignes) avec `pyarrow`, sans charger les données. Chaque mois est comparé à janvier. Une ligne `DIFFÉRENCE` apparaît si une colonne manque ou change de type.

## Lecture et comparaison des schémas

Le code lit le schéma de chaque fichier, le compare à celui de janvier, et affiche le nombre de colonnes et de lignes. Les nombres de lignes serviront ensuite de contrôle après chargement.

In [1]:
from pathlib import Path
import pyarrow.parquet as pq

fichiers = sorted(Path("../data/raw").glob("yellow_tripdata_2024-*.parquet"))
schemas = {f.name: pq.read_schema(f) for f in fichiers}

nom_ref, schemas_ref = next(iter(schemas.items()))
ref = {c.name: str(c.type) for c in schemas_ref}

for f in fichiers:
    actuel = {c.name: str(c.type) for c in schemas[f.name]}
    lignes = pq.read_metadata(f).num_rows
    print(f"{f.name} : {len(actuel)} colonnes, {lignes:,} lignes")
    for col in sorted(set(ref) | set(actuel)):
        if ref.get(col) != actuel.get(col):
            print(f"  DIFFÉRENCE sur {col} : {ref.get(col)} -> {actuel.get(col)}")

yellow_tripdata_2024-01.parquet : 19 colonnes, 2,964,624 lignes
yellow_tripdata_2024-02.parquet : 19 colonnes, 3,007,526 lignes
yellow_tripdata_2024-03.parquet : 19 colonnes, 3,582,628 lignes


## Constat

| Fichier | Colonnes | Lignes |
|---|---|---|
| `yellow_tripdata_2024-01.parquet` | 19 | 2 964 624 |
| `yellow_tripdata_2024-02.parquet` | 19 | 3 007 526 |
| `yellow_tripdata_2024-03.parquet` | 19 | 3 582 628 |

- Aucune ligne `DIFFÉRENCE` : les noms et les types de colonnes sont **identiques** dans les trois fichiers.
- Total attendu en Bronze après chargement : **9 554 778** lignes.
- Le volume varie peu entre janvier et février (environ +1,4 %), puis augmente d'environ 19 % en mars.

## Conclusion

Les trois mois peuvent être chargés dans la même table `bronze.yellow_trips` avec le même script. Les nombres de lignes ci-dessus servent de contrôle de réconciliation après chaque chargement.

## Limites

- Cette comparaison porte sur la **structure** (noms et types), pas sur le contenu. Les valeurs aberrantes et les dates hors du mois sont traitées en Silver, selon `regles_qualite.md`.
- Seuls trois mois sont comparés. Il faudra refaire cette vérification pour chaque nouveau fichier ajouté plus tard.